In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# Schrodinger's equation functions
# ============================================================

def allowed_k(L, m):
    """
    Enforce periodic boundary condition compatibility:
        k = 2*pi*m / L
    """
    return 2 * np.pi * m / L


def hopping_profile(x, t, J, k, omega):
    """
    Bond amplitude on bond x -> x+1:
        a_x(t) = -J cos(k x - omega t)

    The minus sign is included here so the RHS reads naturally.
    """
    return -J * np.cos(k * x - omega * t)


def schrodinger_rhs(t, psi, L, J, k, omega):
    """
    Right-hand side of
        d psi / dt = -i H(t) psi

    using the site-basis nearest-neighbor equation directly.

    For each site x:
        i d/dt psi_x =
            a_{x-1}(t) psi_{x-1}
          + a_x(t)     psi_{x+1}

    where
        a_x(t) = -J cos(kx - omega t)

    with periodic boundary conditions.
    """
    dpsi = np.zeros(L, dtype=np.complex128)

    for x in range(L):
        xm1 = (x - 1) % L
        xp1 = (x + 1) % L

        a_left = hopping_profile(xm1, t, J, k, omega)   # bond (x-1) -> x
        a_right = hopping_profile(x, t, J, k, omega)    # bond x -> (x+1)

        # Since cos(...) is real, Hermitian conjugate is automatically handled
        # by the symmetric coupling to neighbors.
        dpsi[x] = -1j * (a_left * psi[xm1] + a_right * psi[xp1])

    return dpsi

In [ ]:
# ============================================================
# Initial states
# ============================================================

def delta_initial_state(L, x0):
    """
    Fully localized state at site x0.
    """
    psi0 = np.zeros(L, dtype=np.complex128)
    psi0[x0 % L] = 1.0
    return psi0


def gaussian_initial_state(L, x0, sigma=3.0, q0=0.0):
    """
    Gaussian wavepacket on a ring:
        psi_x ~ exp[-(dx)^2 / (2 sigma^2)] * exp(i q0 x)

    where dx is the shortest distance on the ring.
    """
    x = np.arange(L, dtype=float)
    dx = (x - x0 + L / 2.0) % L - L / 2.0
    psi0 = np.exp(-(dx**2) / (2.0 * sigma**2)) * np.exp(1j * q0 * x)
    psi0 = psi0.astype(np.complex128)
    psi0 /= np.linalg.norm(psi0)
    return psi0

In [ ]:
# ============================================================
# Time evolution
# ============================================================
def evolve_state_predictor_corrector(L, J, omega, m, psi0, t_max, n_times=400,
                                     n_substeps=20, renormalize_each_step=True):
    """
    Evolve the state using a predictor-corrector method
    (Heun / explicit trapezoid) built directly from the stencil RHS.

    Parameters
    ----------
    L : int
        Number of sites.
    J : float
        Hopping strength.
    omega : float
        Drive frequency.
    m : int
        Integer setting k = 2*pi*m/L.
    psi0 : ndarray, shape (L,)
        Initial state.
    t_max : float
        Final time.
    n_times : int
        Number of output times.
    n_substeps : int
        Number of predictor-corrector substeps between consecutive output times.
        Increase this if agreement with scipy/RK4 is not good enough.
    renormalize_each_step : bool
        If True, normalize after every corrected step to suppress
        long-time numerical drift.

    Returns
    -------
    t_eval : ndarray, shape (n_times,)
    psi_t : ndarray, shape (n_times, L)
    k : float
    """
    k = allowed_k(L, m)

    psi0 = np.asarray(psi0, dtype=np.complex128)
    if psi0.shape != (L,):
        raise ValueError(f"psi0 must have shape ({L},)")

    t_eval = np.linspace(0.0, t_max, n_times)
    psi_t = np.zeros((n_times, L), dtype=np.complex128)

    psi = psi0.copy()
    psi /= np.linalg.norm(psi)
    psi_t[0] = psi

    if n_times < 2:
        return t_eval, psi_t, k

    dt_out = t_eval[1] - t_eval[0]
    dt = dt_out / n_substeps

    t = t_eval[0]

    for it in range(1, n_times):
        for _ in range(n_substeps):
            # Predictor: forward Euler
            f_n = schrodinger_rhs(t, psi, L, J, k, omega)
            psi_pred = psi + dt * f_n

            # Corrector: trapezoid / Heun step
            f_pred = schrodinger_rhs(t + dt, psi_pred, L, J, k, omega)
            psi = psi + 0.5 * dt * (f_n + f_pred)

            if renormalize_each_step:
                psi /= np.linalg.norm(psi)

            t += dt

        psi_t[it] = psi

    # final cleanup
    norms = np.linalg.norm(psi_t, axis=1, keepdims=True)
    psi_t = psi_t / norms

    return t_eval, psi_t, k


In [ ]:
# ============================================================
# Observables
# ============================================================

def probabilities(psi_t):
    """
    Site probabilities P_x(t) = |psi_x(t)|^2
    """
    return np.abs(psi_t)**2


def norm_vs_time(psi_t):
    """
    Norm at each time.
    """
    return np.sum(np.abs(psi_t)**2, axis=1)


def x_expectation(psi_t):
    """
    Naive <x>. Useful mainly before wrap-around becomes important.
    """
    L = psi_t.shape[1]
    x = np.arange(L, dtype=float)
    prob = probabilities(psi_t)
    return prob @ x


def x2_expectation(psi_t):
    """
    Naive <x^2>.
    """
    L = psi_t.shape[1]
    x = np.arange(L, dtype=float)
    prob = probabilities(psi_t)
    return prob @ (x**2)


def width_vs_time(psi_t):
    """
    Naive width sqrt(<x^2> - <x>^2).
    """
    x1 = x_expectation(psi_t)
    x2 = x2_expectation(psi_t)
    var = np.maximum(x2 - x1**2, 0.0)
    return np.sqrt(var)


def total_current(psi_t, t_eval, L, J, omega, m):
    """
    Total bond current:
        I(t) = sum_x j_x(t)

    with
        j_x(t) = 2 Im[ a_x(t) psi_x^*(t) psi_{x+1}(t) ]
    where
        a_x(t) = -J cos(kx - omega t)
    """
    k = allowed_k(L, m)
    currents = np.zeros(len(t_eval), dtype=float)

    for it, t in enumerate(t_eval):
        psi = psi_t[it]
        total_j = 0.0
        for x in range(L):
            xp1 = (x + 1) % L
            a_x = hopping_profile(x, t, J, k, omega)
            total_j += 2.0 * np.imag(a_x * np.conjugate(psi[x]) * psi[xp1])
        currents[it] = total_j

    return currents

In [ ]:
# ============================================================
# Plotting
# ============================================================

def plot_probability_snapshot(psi_t, time_index, title=None):
    """
    Plot P_x at one selected time.
    Since x is discrete, we plot probability site by site.
    """
    prob = np.abs(psi_t[time_index])**2
    x = np.arange(len(prob))

    plt.figure(figsize=(8, 4))
    plt.plot(x, prob, marker='o', linestyle='-', ms=3)
    plt.xlabel("site x")
    plt.ylabel(r"$|\psi_x|^2$")
    if title is None:
        title = f"Probability snapshot at index {time_index}"
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_density_map(t_eval, psi_t, vmax=None):
    """
    Heatmap of P_x(t) = |psi_x(t)|^2.
    """
    prob = probabilities(psi_t)
    L = psi_t.shape[1]

    plt.figure(figsize=(10, 5))
    plt.imshow(
        prob.T,
        aspect='auto',
        origin='lower',
        extent=[t_eval[0], t_eval[-1], 0, L - 1],
        vmax=vmax
    )
    plt.colorbar(label=r"$|\psi_x(t)|^2$")
    plt.xlabel("time t")
    plt.ylabel("site x")
    plt.title("Density map on the ring")
    plt.tight_layout()
    plt.show()


def plot_basic_observables(t_eval, psi_t, L, J, omega, m):
    """
    Plot norm, naive <x>, width, and total current.
    """
    norms = norm_vs_time(psi_t)
    x_mean = x_expectation(psi_t)
    widths = width_vs_time(psi_t)
    currents = total_current(psi_t, t_eval, L, J, omega, m)

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))

    axes[0, 0].plot(t_eval, norms)
    axes[0, 0].set_title("Norm")
    axes[0, 0].set_xlabel("t")
    axes[0, 0].set_ylabel(r"$\langle \psi | \psi \rangle$")

    axes[0, 1].plot(t_eval, x_mean)
    axes[0, 1].set_title(r"Naive $\langle x \rangle$")
    axes[0, 1].set_xlabel("t")
    axes[0, 1].set_ylabel(r"$\langle x \rangle$")

    axes[1, 0].plot(t_eval, widths)
    axes[1, 0].set_title("Naive width")
    axes[1, 0].set_xlabel("t")
    axes[1, 0].set_ylabel("width")

    axes[1, 1].plot(t_eval, currents)
    axes[1, 1].set_title("Total current")
    axes[1, 1].set_xlabel("t")
    axes[1, 1].set_ylabel("I(t)")

    plt.tight_layout()
    plt.show()

In [ ]:
L = 200          # system size
J = 5.0          # hopping scale
omega = 2     # drive frequency
m = 80          # sets k = 2*pi*m/L

t_max = 100
n_times = 500

k = allowed_k(L, m)
# T = 2.0 * np.pi / omega

print(f"L = {L}")
print(f"J = {J}")
print(f"omega = {omega}")
print(f"m = {m}")
print(f"k = 2*pi*m/L = {k:.6f}")
print(f"Drive period T = {T:.6f}")

# --------------------------------------------------------
# Choose initial state
# --------------------------------------------------------

# Option 1: delta-function localized at one site
x0_delta = L // 2
psi0_delta = delta_initial_state(L, x0_delta)

# Option 2: Gaussian packet
x0_gauss = L // 2
sigma = 5
q0 = 0.0
psi0_gauss = gaussian_initial_state(L, x0_gauss, sigma=sigma, q0=q0)

# --------------------------------------------------------
# Evolve one of them
# --------------------------------------------------------
use_gaussian = False


if use_gaussian:
    psi0 = psi0_gauss
    init_name = f"Gaussian packet (x0={x0_gauss}, sigma={sigma}, q0={q0})"
else:
    psi0 = psi0_delta
    init_name = f"Delta state at site {x0_delta}"

print(f"Initial state: {init_name}")

t_eval, psi_t, k = evolve_state_predictor_corrector(
    L=L,
    J=J,
    omega=omega,
    m=m,
    psi0=psi0,
    t_max=t_max,
    n_times=n_times
)

# --------------------------------------------------------
# Plots
# --------------------------------------------------------
plot_probability_snapshot(psi_t, 0, title=f"Initial snapshot: {init_name}")
plot_probability_snapshot(psi_t, len(t_eval)//2, title="Mid-time snapshot")
plot_probability_snapshot(psi_t, -1, title="Final snapshot")

plot_density_map(t_eval, psi_t)
plot_basic_observables(t_eval, psi_t, L, J, omega, m)